# Project 1: Retail Sales Intelligence - Exploratory Data Analysis
**Author:** MARAGATHALAKSHMI B | Data Analyst

This notebook conducts automated data ingestion, data cleaning, descriptive statistics, outlier detection, and multidimensional sales performance analysis.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid')
print('Libraries imported successfully.')


## 1. Load Raw Dataset & Audit Quality


In [1]:
df_raw = pd.read_csv('../datasets/retail_sales_raw.csv')
print(f'Shape of raw dataset: {df_raw.shape}')
df_raw.info()


## 2. Data Cleaning & Pipeline Transformations


In [1]:
# Handle whitespace in categorical columns
for col in ['region', 'category', 'sub_category', 'segment']:
    if col in df_raw.columns and df_raw[col].dtype == 'object':
        df_raw[col] = df_raw[col].astype(str).str.strip().str.title()

# Standardize dates
df_raw['order_date'] = pd.to_datetime(df_raw['order_date'])
df_raw['ship_date'] = pd.to_datetime(df_raw['ship_date'])

# Drop duplicates and fill missing profits
df_clean = df_raw.drop_duplicates()
df_clean['profit'] = pd.to_numeric(df_clean['profit'], errors='coerce')
df_clean['profit'] = df_clean['profit'].fillna(df_clean.groupby('sub_category')['profit'].transform('median'))

print(f'Cleaned dataset shape: {df_clean.shape}')
print(f'Missing values remaining: {df_clean.isnull().sum().sum()}')


## 3. Revenue & Profit Margin Summary


In [1]:
summary = df_clean.groupby('category').agg(
    Total_Sales=('sales', 'sum'),
    Total_Profit=('profit', 'sum'),
    Avg_Discount=('discount', 'mean'),
    Order_Count=('order_id', 'count')
).reset_index()

summary['Profit_Margin_%'] = (summary['Total_Profit'] / summary['Total_Sales']) * 100
print(summary.to_string(index=False))


## 4. Key Business Insights
- Technology generates the highest revenue with solid profit margins.
- Discounts above 20% severely erode margin and lead to negative profitability in furniture sub-categories.
